# Semanic model

## **Google Drive intialization**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/Fake_Job_Detection"

os.makedirs(f"{project_path}/embeddings", exist_ok=True)
os.makedirs(f"{project_path}/models", exist_ok=True)

print("Project folders created successfully!")
print(project_path)

Project folders created successfully!
/content/drive/MyDrive/Fake_Job_Detection


## model intialization

### model installation

In [ ]:
!pip install -q sentence-transformers

### import

In [ ]:
from sentence_transformers import SentenceTransformer

### Load the Semantic Embedding Model

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Data Transformation


In [ ]:
 import pandas as pd

df = pd.read_csv(
    "/content/fake_job_core_clean_FIXED.csv",
    engine="python"
)

print("Dataset shape:", df.shape)
print(df.head(5))

Dataset shape: (15870, 29)
   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                   

## Train , Validation , Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["fraudulent"])
y = df["fraudulent"]

# Same split used in the TF-IDF baseline
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Testing set:", X_test.shape)

print("\nFraud percentages:")
print("Train:", round(y_train.mean() * 100, 2), "%")
print("Validation:", round(y_val.mean() * 100, 2), "%")
print("Test:", round(y_test.mean() * 100, 2), "%")

Training set: (11109, 28)
Validation set: (2380, 28)
Testing set: (2381, 28)

Fraud percentages:
Train: 4.48 %
Validation: 4.45 %
Test: 4.49 %


In [15]:
print(X_train["structured_text"].iloc[0])

Title: Sales Coordinator | Company: BCV is the premier full-service social media provider in the hospitality industry. Our social media experts have developed proprietary services, methodologies, and tools to continually create significant value for clients. We are the only provider with the capacity to monitor the social space and manage potential crises 24 hours a day, 7 days a week. Our comprehensive management services are supported by a robust arsenal of cutting-edge tools and analytics designed to drive traffic, increase sales, create unforgettable experiences for guests, and mitigate negative customer experiences. | Description: As the premier social media provider to luxury hotels and resorts, BCV is uniquely positioned for rapid, uninhibited growth. This role is the primary support to the sales team that aids in the growth and success of BCV. This will be achieved by working with a talented team of sales representatives, understanding client’s business needs, and how BCV can a

###

## Generate Semantic Embeddings

In [18]:
import numpy as np

train_path = "/content/drive/MyDrive/Fake_Job_Detection/embeddings/X_train_embeddings.npy"

np.save(train_path, X_train_embeddings)

print("Training embeddings saved successfully!")
print("Shape:", X_train_embeddings.shape)
print("Location:", train_path)

Training embeddings saved successfully!
Shape: (11109, 384)
Location: /content/drive/MyDrive/Fake_Job_Detection/embeddings/X_train_embeddings.npy


In [17]:
X_train_embeddings = embedding_model.encode(
    X_train["structured_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Training embeddings shape:", X_train_embeddings.shape)

Batches:   0%|          | 0/348 [00:00<?, ?it/s]

Training embeddings shape: (11109, 384)


## Generate Validation Embeddings

In [19]:
X_val_embeddings = embedding_model.encode(
    X_val["structured_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Validation embeddings shape:", X_val_embeddings.shape)

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Validation embeddings shape: (2380, 384)


In [20]:
import numpy as np

val_path = "/content/drive/MyDrive/Fake_Job_Detection/embeddings/X_val_embeddings.npy"

np.save(val_path, X_val_embeddings)

print("Validation embeddings saved successfully!")
print("Shape:", X_val_embeddings.shape)
print("Location:", val_path)

Validation embeddings saved successfully!
Shape: (2380, 384)
Location: /content/drive/MyDrive/Fake_Job_Detection/embeddings/X_val_embeddings.npy


## Generate Test Embeddings

In [ ]:
X_test_embeddings = embedding_model.encode(
    X_test["structured_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Testing embeddings shape:", X_test_embeddings.shape)

In [ ]:
import numpy as np

test_path = "/content/drive/MyDrive/Fake_Job_Detection/embeddings/X_test_embeddings.npy"

np.save(test_path, X_test_embeddings)

print("Test embeddings saved successfully!")
print("Shape:", X_test_embeddings.shape)
print("Location:", test_path)

## Semantic Embedding Classifier

###  Logistic Regression on Semantic Embeddings

In [22]:
from sklearn.linear_model import LogisticRegression

semantic_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

semantic_lr.fit(X_train_embeddings, y_train)

print("Semantic Logistic Regression trained successfully!")

Semantic Logistic Regression trained successfully!


### Validation Predictions

In [23]:
y_val_pred_semantic = semantic_lr.predict(X_val_embeddings)
y_val_prob_semantic = semantic_lr.predict_proba(X_val_embeddings)[:, 1]

print("Validation predictions generated!")
print("Predictions:", len(y_val_pred_semantic))
print("Fraud probabilities:", len(y_val_prob_semantic))

Validation predictions generated!
Predictions: 2380
Fraud probabilities: 2380


### 10.10.3 Semantic Model Evaluation

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

semantic_accuracy = accuracy_score(y_val, y_val_pred_semantic)
semantic_precision = precision_score(y_val, y_val_pred_semantic)
semantic_recall = recall_score(y_val, y_val_pred_semantic)
semantic_f1 = f1_score(y_val, y_val_pred_semantic)
semantic_pr_auc = average_precision_score(y_val, y_val_prob_semantic)

print("Semantic Embeddings + Logistic Regression")
print("------------------------------------------")
print(f"Accuracy : {semantic_accuracy:.4f}")
print(f"Precision: {semantic_precision:.4f}")
print(f"Recall   : {semantic_recall:.4f}")
print(f"F1 Score : {semantic_f1:.4f}")
print(f"PR-AUC   : {semantic_pr_auc:.4f}")

Semantic Embeddings + Logistic Regression
------------------------------------------
Accuracy : 0.8761
Precision: 0.2411
Recall   : 0.8302
F1 Score : 0.3737
PR-AUC   : 0.5290


### 10.10.4 Semantic Model Confusion Matrix

In [25]:
from sklearn.metrics import confusion_matrix, classification_report

cm_semantic = confusion_matrix(y_val, y_val_pred_semantic)

print("Confusion Matrix:")
print(cm_semantic)

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_semantic,
    target_names=["Legitimate", "Fraudulent"]
))

Confusion Matrix:
[[1997  277]
 [  18   88]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      0.88      0.93      2274
  Fraudulent       0.24      0.83      0.37       106

    accuracy                           0.88      2380
   macro avg       0.62      0.85      0.65      2380
weighted avg       0.96      0.88      0.91      2380



### 10.10.5 Semantic Model Threshold Analysis

In [26]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.91, 0.05)

print("Threshold | Precision | Recall | F1")
print("-" * 40)

for threshold in thresholds:
    y_threshold_pred = (y_val_prob_semantic >= threshold).astype(int)

    precision = precision_score(y_val, y_threshold_pred, zero_division=0)
    recall = recall_score(y_val, y_threshold_pred, zero_division=0)
    f1 = f1_score(y_val, y_threshold_pred, zero_division=0)

    print(
        f"{threshold:8.2f} | "
        f"{precision:9.4f} | "
        f"{recall:6.4f} | "
        f"{f1:6.4f}"
    )

Threshold | Precision | Recall | F1
----------------------------------------
    0.10 |    0.0848 | 0.9906 | 0.1562
    0.15 |    0.1043 | 0.9811 | 0.1886
    0.20 |    0.1204 | 0.9528 | 0.2138
    0.25 |    0.1367 | 0.9528 | 0.2391
    0.30 |    0.1510 | 0.9245 | 0.2596
    0.35 |    0.1641 | 0.8962 | 0.2774
    0.40 |    0.1898 | 0.8774 | 0.3121
    0.45 |    0.2188 | 0.8774 | 0.3503
    0.50 |    0.2411 | 0.8302 | 0.3737
    0.55 |    0.2716 | 0.8019 | 0.4057
    0.60 |    0.3043 | 0.7925 | 0.4398
    0.65 |    0.3292 | 0.7547 | 0.4585
    0.70 |    0.3641 | 0.7075 | 0.4808
    0.75 |    0.4132 | 0.6509 | 0.5055
    0.80 |    0.4493 | 0.5849 | 0.5082
    0.85 |    0.5429 | 0.5377 | 0.5403
    0.90 |    0.6111 | 0.4151 | 0.4944


## 11. Neural Network Classifier
### 11.1 MLP Architecture

In [27]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [28]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

TensorFlow version: 2.20.0
GPU available: False


### MLP


In [29]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

mlp_model = Sequential([
    Dense(128, activation="relu", input_shape=(384,)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

mlp_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,601 (225.00 KB)

 Trainable params: 57,601 (225.00 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
from tensorflow.keras.optimizers import Adam

mlp_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

print("MLP compiled successfully!")

MLP compiled successfully!


### training mlp

In [31]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights_array))

print("Class weights:")
print(class_weights)

Class weights:
{np.int64(0): np.float64(0.523466214305909), np.int64(1): np.float64(11.153614457831326)}


In [32]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

history = mlp_model.fit(
    X_train_embeddings,
    y_train,
    validation_data=(X_val_embeddings, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

print("MLP training completed!")

Epoch 1/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 21s 28ms/step - loss: 0.4981 - pr_auc: 0.2762 - precision: 0.1341 - recall: 0.7932 - val_loss: 0.4271 - val_pr_auc: 0.5464 - val_precision: 0.1670 - val_recall: 0.8491
Epoch 2/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3571 - pr_auc: 0.4611 - precision: 0.2047 - recall: 0.8494 - val_loss: 0.4845 - val_pr_auc: 0.6463 - val_precision: 0.1523 - val_recall: 0.9151
Epoch 3/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2931 - pr_auc: 0.5885 - precision: 0.2393 - recall: 0.8835 - val_loss: 0.2586 - val_pr_auc: 0.7109 - val_precision: 0.2644 - val_recall: 0.8679
Epoch 4/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.2415 - pr_auc: 0.6936 - precision: 0.2945 - recall: 0.9036 - val_loss: 0.2446 - val_pr_auc: 0.7515 - val_precision: 0.2738 - val_recall: 0.8962
Epoch 5/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1999 - pr_auc: 0.7518 - precision: 0.3434 - recall: 0.9317 - val_loss: 0.1669 - val_pr_auc: 0.7948 - val_precisi

### MLP Validation Predictions

In [33]:
y_val_prob_mlp = mlp_model.predict(
    X_val_embeddings,
    verbose=0
).ravel()

y_val_pred_mlp = (y_val_prob_mlp >= 0.5).astype(int)

print("Validation predictions generated!")
print("Predictions:", len(y_val_pred_mlp))
print("Fraud probabilities:", len(y_val_prob_mlp))

Validation predictions generated!
Predictions: 2380
Fraud probabilities: 2380


### 11.2 MLP Validation Evaluation

In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

mlp_accuracy = accuracy_score(y_val, y_val_pred_mlp)
mlp_precision = precision_score(y_val, y_val_pred_mlp, zero_division=0)
mlp_recall = recall_score(y_val, y_val_pred_mlp, zero_division=0)
mlp_f1 = f1_score(y_val, y_val_pred_mlp, zero_division=0)
mlp_pr_auc = average_precision_score(y_val, y_val_prob_mlp)

print("Semantic Embeddings + MLP")
print("-------------------------")
print(f"Accuracy : {mlp_accuracy:.4f}")
print(f"Precision: {mlp_precision:.4f}")
print(f"Recall   : {mlp_recall:.4f}")
print(f"F1 Score : {mlp_f1:.4f}")
print(f"PR-AUC   : {mlp_pr_auc:.4f}")

Semantic Embeddings + MLP
-------------------------
Accuracy : 0.9811
Precision: 0.7748
Recall   : 0.8113
F1 Score : 0.7926
PR-AUC   : 0.8563


### Confusion matrix

In [35]:
from sklearn.metrics import confusion_matrix, classification_report

cm_mlp = confusion_matrix(y_val, y_val_pred_mlp)

print("MLP Confusion Matrix")
print("--------------------")
print(cm_mlp)

print("\nClassification Report")
print("----------------------")
print(classification_report(
    y_val,
    y_val_pred_mlp,
    target_names=["Legitimate", "Fraudulent"],
    digits=4
))

MLP Confusion Matrix
--------------------
[[2249   25]
 [  20   86]]

Classification Report
----------------------
              precision    recall  f1-score   support

  Legitimate     0.9912    0.9890    0.9901      2274
  Fraudulent     0.7748    0.8113    0.7926       106

    accuracy                         0.9811      2380
   macro avg     0.8830    0.9002    0.8914      2380
weighted avg     0.9815    0.9811    0.9813      2380



In [36]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Semantic Embeddings + Logistic Regression",
        "Semantic Embeddings + MLP"
    ],
    "Accuracy": [
        0.9735,
        0.8761,
        mlp_accuracy
    ],
    "Precision": [
        0.6503,
        0.2411,
        mlp_precision
    ],
    "Recall": [
        0.8774,
        0.8302,
        mlp_recall
    ],
    "F1 Score": [
        0.7470,
        0.3737,
        mlp_f1
    ],
    "PR-AUC": [
        0.8805,
        0.5290,
        mlp_pr_auc
    ]
})

print(comparison.to_string(index=False))

                                    Model  Accuracy  Precision   Recall  F1 Score   PR-AUC
             TF-IDF + Logistic Regression  0.973500   0.650300 0.877400  0.747000 0.880500
Semantic Embeddings + Logistic Regression  0.876100   0.241100 0.830200  0.373700 0.529000
                Semantic Embeddings + MLP  0.981092   0.774775 0.811321  0.792627 0.856311


## final testing

In [52]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

test_accuracy = accuracy_score(y_test, y_test_pred_mlp)
test_precision = precision_score(y_test, y_test_pred_mlp, zero_division=0)
test_recall = recall_score(y_test, y_test_pred_mlp, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred_mlp, zero_division=0)
test_pr_auc = average_precision_score(y_test, y_test_prob_mlp)

print("FINAL TEST RESULTS — Semantic Embeddings + MLP")
print("==============================================")
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1 Score : {test_f1:.4f}")
print(f"PR-AUC   : {test_pr_auc:.4f}")

print("\nConfusion Matrix")
print("----------------")
print(confusion_matrix(y_test, y_test_pred_mlp))

print("\nClassification Report")
print("---------------------")
print(classification_report(
    y_test,
    y_test_pred_mlp,
    target_names=["Legitimate", "Fraudulent"],
    digits=4
))

FINAL TEST RESULTS — Semantic Embeddings + MLP
Accuracy : 0.9744
Precision: 0.7054
Recall   : 0.7383
F1 Score : 0.7215
PR-AUC   : 0.7955

Confusion Matrix
----------------
[[2241   33]
 [  28   79]]

Classification Report
---------------------
              precision    recall  f1-score   support

  Legitimate     0.9877    0.9855    0.9866      2274
  Fraudulent     0.7054    0.7383    0.7215       107

    accuracy                         0.9744      2381
   macro avg     0.8465    0.8619    0.8540      2381
weighted avg     0.9750    0.9744    0.9747      2381



In [55]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

tfidf_test_accuracy = accuracy_score(y_test, y_test_pred_tfidf)
tfidf_test_precision = precision_score(y_test, y_test_pred_tfidf, zero_division=0)
tfidf_test_recall = recall_score(y_test, y_test_pred_tfidf, zero_division=0)
tfidf_test_f1 = f1_score(y_test, y_test_pred_tfidf, zero_division=0)
tfidf_test_pr_auc = average_precision_score(y_test, y_test_prob_tfidf)

print("FINAL TEST RESULTS — TF-IDF + Logistic Regression")
print("===================================================")
print(f"Accuracy : {tfidf_test_accuracy:.4f}")
print(f"Precision: {tfidf_test_precision:.4f}")
print(f"Recall   : {tfidf_test_recall:.4f}")
print(f"F1 Score : {tfidf_test_f1:.4f}")
print(f"PR-AUC   : {tfidf_test_pr_auc:.4f}")

print("\nConfusion Matrix")
print("----------------")
print(confusion_matrix(y_test, y_test_pred_tfidf))

print("\nClassification Report")
print("---------------------")
print(classification_report(
    y_test,
    y_test_pred_tfidf,
    target_names=["Legitimate", "Fraudulent"],
    digits=4
))

FINAL TEST RESULTS — TF-IDF + Logistic Regression
Accuracy : 0.9685
Precision: 0.6053
Recall   : 0.8598
F1 Score : 0.7104
PR-AUC   : 0.8582

Confusion Matrix
----------------
[[2214   60]
 [  15   92]]

Classification Report
---------------------
              precision    recall  f1-score   support

  Legitimate     0.9933    0.9736    0.9833      2274
  Fraudulent     0.6053    0.8598    0.7104       107

    accuracy                         0.9685      2381
   macro avg     0.7993    0.9167    0.8469      2381
weighted avg     0.9758    0.9685    0.9711      2381



## baseline classifier

In [50]:
print("MLP model available:", 'mlp_model' in globals())

MLP model available: True


In [48]:
import numpy as np

embedding_path = "/content/drive/MyDrive/Fake_Job_Detection/embeddings"

X_test_embeddings = np.load(
    f"{embedding_path}/X_test_embeddings.npy"
)

print("Test embeddings loaded successfully.")
print("Shape:", X_test_embeddings.shape)

Test embeddings loaded successfully.
Shape: (2381, 384)


In [44]:
import numpy as np

embedding_path = "/content/drive/MyDrive/Fake_Job_Detection/embeddings"

X_train_embeddings = np.load(
    f"{embedding_path}/X_train_embeddings.npy"
)

X_val_embeddings = np.load(
    f"{embedding_path}/X_val_embeddings.npy"
)

X_test_embeddings = np.load(
    f"{embedding_path}/X_test_embeddings.npy"
)

print("Embeddings loaded successfully.")
print("Train:", X_train_embeddings.shape)
print("Validation:", X_val_embeddings.shape)
print("Test:", X_test_embeddings.shape)

Embeddings loaded successfully.
Train: (11109, 384)
Validation: (2380, 384)
Test: (2381, 384)


In [51]:
y_test_prob_mlp = mlp_model.predict(
    X_test_embeddings,
    verbose=0
).ravel()

y_test_pred_mlp = (y_test_prob_mlp >= 0.5).astype(int)

print("MLP test predictions generated successfully.")
print("Number of test samples:", len(y_test_pred_mlp))

MLP test predictions generated successfully.
Number of test samples: 2381


In [45]:
y_test_prob_mlp = mlp_model.predict(
    X_test_embeddings,
    verbose=0
).ravel()

y_test_pred_mlp = (y_test_prob_mlp >= 0.5).astype(int)

print("MLP test predictions generated successfully.")
print("Number of test samples:", len(y_test_pred_mlp))

MLP test predictions generated successfully.
Number of test samples: 2381


In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train["structured_text"])
X_test_tfidf = tfidf.transform(X_test["structured_text"])

print("TF-IDF restored successfully.")
print("Training shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

TF-IDF restored successfully.
Training shape: (11109, 20000)
Test shape: (2381, 20000)


In [53]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

baseline_model.fit(X_train_tfidf, y_train)

print("TF-IDF + Logistic Regression baseline restored.")

TF-IDF + Logistic Regression baseline restored.


In [54]:
y_test_pred_tfidf = baseline_model.predict(X_test_tfidf)
y_test_prob_tfidf = baseline_model.predict_proba(X_test_tfidf)[:, 1]

print("TF-IDF test predictions generated successfully.")
print("Number of test samples:", len(y_test_pred_tfidf))

TF-IDF test predictions generated successfully.
Number of test samples: 2381


## Google Drive

In [56]:
# ============================================================
# SAVE COMPLETE FAKE JOB DETECTION PROJECT
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib

# ------------------------------------------------------------
# 1. Project directories
# ------------------------------------------------------------

BASE_PATH = "/content/drive/MyDrive/Fake_Job_Detection"

MODEL_PATH = os.path.join(BASE_PATH, "models")
EMBEDDING_PATH = os.path.join(BASE_PATH, "embeddings")
RESULT_PATH = os.path.join(BASE_PATH, "results")

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(EMBEDDING_PATH, exist_ok=True)
os.makedirs(RESULT_PATH, exist_ok=True)

print("Project directories ready.")


# ------------------------------------------------------------
# 2. Save Semantic MLP
# ------------------------------------------------------------

mlp_model.save(
    os.path.join(MODEL_PATH, "semantic_mlp_final.keras")
)

print("✓ Semantic MLP saved")


# ------------------------------------------------------------
# 3. Save TF-IDF Vectorizer
# ------------------------------------------------------------

joblib.dump(
    tfidf,
    os.path.join(MODEL_PATH, "tfidf_vectorizer.joblib")
)

print("✓ TF-IDF vectorizer saved")


# ------------------------------------------------------------
# 4. Save TF-IDF Logistic Regression
# ------------------------------------------------------------

joblib.dump(
    baseline_model,
    os.path.join(MODEL_PATH, "tfidf_logistic_regression.joblib")
)

print("✓ TF-IDF Logistic Regression saved")


# ------------------------------------------------------------
# 5. Save embeddings
# ------------------------------------------------------------

np.save(
    os.path.join(EMBEDDING_PATH, "X_train_embeddings.npy"),
    X_train_embeddings
)

np.save(
    os.path.join(EMBEDDING_PATH, "X_val_embeddings.npy"),
    X_val_embeddings
)

np.save(
    os.path.join(EMBEDDING_PATH, "X_test_embeddings.npy"),
    X_test_embeddings
)

print("✓ Train/validation/test embeddings saved")


# ------------------------------------------------------------
# 6. Save model comparison results
# ------------------------------------------------------------

comparison.to_csv(
    os.path.join(RESULT_PATH, "model_comparison_validation.csv"),
    index=False
)

print("✓ Validation comparison saved")


# ------------------------------------------------------------
# 7. Save final test results
# ------------------------------------------------------------

final_test_results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Semantic Embeddings + MLP"
    ],
    "Accuracy": [
        tfidf_test_accuracy,
        test_accuracy
    ],
    "Precision": [
        tfidf_test_precision,
        test_precision
    ],
    "Recall": [
        tfidf_test_recall,
        test_recall
    ],
    "F1 Score": [
        tfidf_test_f1,
        test_f1
    ],
    "PR-AUC": [
        tfidf_test_pr_auc,
        test_pr_auc
    ]
})

final_test_results.to_csv(
    os.path.join(RESULT_PATH, "final_test_results.csv"),
    index=False
)

print("✓ Final test results saved")


# ------------------------------------------------------------
# 8. Save dataset split indices
# ------------------------------------------------------------

np.save(
    os.path.join(RESULT_PATH, "train_indices.npy"),
    X_train.index.to_numpy()
)

np.save(
    os.path.join(RESULT_PATH, "validation_indices.npy"),
    X_val.index.to_numpy()
)

np.save(
    os.path.join(RESULT_PATH, "test_indices.npy"),
    X_test.index.to_numpy()
)

print("✓ Dataset split indices saved")


# ------------------------------------------------------------
# 9. Save project information
# ------------------------------------------------------------

project_info = {
    "dataset_rows": len(df),
    "dataset_columns": len(df.columns),
    "train_samples": len(X_train),
    "validation_samples": len(X_val),
    "test_samples": len(X_test),
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_dimensions": 384,
    "tfidf_features": 20000,
    "random_state": 42,
    "final_proposed_model": "Semantic Embeddings + MLP",
    "baseline_model": "TF-IDF + Logistic Regression"
}

pd.Series(project_info).to_json(
    os.path.join(RESULT_PATH, "project_info.json"),
    indent=4
)

print("✓ Project information saved")


# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("COMPLETE PROJECT BACKUP FINISHED")
print("=" * 60)

print(f"\nProject location:")
print(BASE_PATH)

print("\nSaved folders:")
print("├── models/")
print("├── embeddings/")
print("└── results/")

print("\nEverything important has been saved to Google Drive.")

Project directories ready.
✓ Semantic MLP saved
✓ TF-IDF vectorizer saved
✓ TF-IDF Logistic Regression saved
✓ Train/validation/test embeddings saved
✓ Validation comparison saved
✓ Final test results saved
✓ Dataset split indices saved
✓ Project information saved

COMPLETE PROJECT BACKUP FINISHED

Project location:
/content/drive/MyDrive/Fake_Job_Detection

Saved folders:
├── models/
├── embeddings/
└── results/

Everything important has been saved to Google Drive.


In [57]:
# ============================================================
# SAVE SENTENCE TRANSFORMER FOR INFERENCE
# ============================================================

import os

transformer_path = (
    "/content/drive/MyDrive/"
    "Fake_Job_Detection/models/all-MiniLM-L6-v2"
)

os.makedirs(transformer_path, exist_ok=True)

embedding_model.save(transformer_path)

print("Sentence Transformer saved successfully.")
print("Model:", "all-MiniLM-L6-v2")
print("Location:", transformer_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence Transformer saved successfully.
Model: all-MiniLM-L6-v2
Location: /content/drive/MyDrive/Fake_Job_Detection/models/all-MiniLM-L6-v2


##  INFERENCE PIPELINE

In [58]:


import numpy as np

def predict_job_fraud(
    title="",
    company_profile="",
    description="",
    requirements="",
    benefits="",
    employment_type="",
    location="",
    salary_range="",
    industry="",
    function=""
):
    """
    Predict whether a job posting is potentially fraudulent.

    Returns:
        fraud_probability: probability of fraud
        prediction: 0 = Legitimate, 1 = Fraudulent
    """

    # --------------------------------------------------------
    # 1. Create the same structured text format used in training
    # --------------------------------------------------------

    structured_text = (
        f"Title: {title} | "
        f"Company: {company_profile} | "
        f"Description: {description} | "
        f"Requirements: {requirements} | "
        f"Benefits: {benefits} | "
        f"Employment Type: {employment_type} | "
        f"Location: {location} | "
        f"Salary: {salary_range} | "
        f"Industry: {industry} | "
        f"Function: {function}"
    )

    # --------------------------------------------------------
    # 2. Generate semantic embedding
    # --------------------------------------------------------

    embedding = embedding_model.encode(
        [structured_text],
        convert_to_numpy=True
    )

    # --------------------------------------------------------
    # 3. Predict fraud probability
    # --------------------------------------------------------

    fraud_probability = float(
        mlp_model.predict(embedding, verbose=0)[0][0]
    )

    # --------------------------------------------------------
    # 4. Convert probability to class
    # --------------------------------------------------------

    prediction = int(fraud_probability >= 0.5)

    return {
        "fraud_probability": fraud_probability,
        "prediction": prediction
    }


print("Inference function created successfully.")

Inference function created successfully.


### Test the inference pipeline

In [59]:
# ============================================================
# TEST INFERENCE PIPELINE
# ============================================================

result = predict_job_fraud(
    title="Marketing Intern",
    company_profile="A growing digital marketing company focused on helping small businesses.",
    description=(
        "We are looking for a motivated marketing intern to assist "
        "with social media, content creation, market research and "
        "basic campaign analysis."
    ),
    requirements=(
        "Good communication skills, basic knowledge of digital marketing, "
        "ability to work in a team and willingness to learn."
    ),
    benefits="Certificate, mentorship and flexible working hours.",
    employment_type="Internship",
    location="Chandigarh, India",
    salary_range="₹10,000 per month",
    industry="Marketing",
    function="Marketing"
)

print("Prediction Result")
print("=================")
print(f"Fraud Probability: {result['fraud_probability']:.4f}")
print(f"Fraud Probability: {result['fraud_probability'] * 100:.2f}%")

if result["prediction"] == 1:
    print("Prediction: POTENTIALLY FRAUDULENT")
else:
    print("Prediction: LIKELY LEGITIMATE")

Prediction Result
Fraud Probability: 0.0004
Fraud Probability: 0.04%
Prediction: LIKELY LEGITIMATE
